In [ ]:
import leafmap
from osgeo import gdal 
import os
import geopandas as gpd
import rasterio
import pandas as pd
import numpy as np
import cv2
import matplotlib.pyplot as plt
import re
from datetime import datetime

In [ ]:
raw_path = os.getcwd()
viirs_denoise_path = raw_path + r'\Download\viirs_avg_rad\background_denoise\denoise'
files_in_directory = os.listdir(viirs_denoise_path)
tif_files = [file for file in files_in_directory if file.lower().endswith('.tif')]
pattern = r'\d{8}'  # Regex pattern to extract dates
dates = []

### Identification of Persistent Nighttime Lights Using the PFM Method

This notebook implements the **Patches Filtering Method (PFM)** based on the study by Yuan et al., (2019)* to enhance the identification of nighttime lights associated with settlements in VIIRS monthly data.

#### Method Steps:
3. **Identification of Recurrent Patches:** Connected component analysis is applied to identify areas of persistent light in multiple monthly images each years.
4. **Persistence Ratio (PR) Calculation:** The frequency of a patch appearing in monthly data is measured, with a 40% threshold set to classify a patch as persistent per years.
5. **Final Image Generation:** Data is filtered to retain only nighttime lights associated with human settlements, and the final image is saved.

This approach reduces noise in VIIRS data, eliminating transient events such as wildfires or reflections from non-urban sources, improving the interpretation of human presence in a given area.

Data are in folder **output_post-processed**

*Yuan, X., Jia, L., Menenti, M., Zhou, J., & Chen, Q. (2019). Filtering the NPP-VIIRS nighttime light data for improved detection of settlements in Africa. Remote Sensing, 11(24), 3002. 

In [ ]:
for file_name in tif_files:
    matches = re.findall(pattern, file_name)
    if matches:
        date = matches[0]  # Take the first match
        dates.append(date)
reformatted_dates = []

# Iterate through dates and reformat them
for date in dates:
    # Reformat the date as "YYYY MM DD"
    reformatted_date = f"{date[:4]} {date[4:6]} {date[6:8]}"
    reformatted_dates.append(reformatted_date)

datetime_dates = []

# Convert dates to datetime objects
for date in dates:
    date_datetime = datetime.strptime(date, "%Y%m%d")
    datetime_dates.append(date_datetime)

years = list(range(2012, 2023))

for year in years:
    year_positions = [i for i, date in enumerate(datetime_dates) if date.year == year]
    files = tif_files[year_positions[0]:year_positions[-1] + 1]

    # FIRST, WE USE THE 8-CONNECTED COMPONENTS METHOD ON EACH MONTH OF A YEAR
    threshold = 0  # Adjust this threshold based on your decimal values
    recurring_patches = None

    for file in files:
        image_path = os.path.join(viirs_denoise_path, file)
        with rasterio.open(image_path) as src:
            viirs_image = src.read(1)  # Read band 1 as a NumPy array
            profile = src.profile
        
        binary_mask = (viirs_image > threshold).astype(np.uint8)
        
        # FIND RECURRING PATCHES IN A YEAR THAT APPEAR MORE THAN TWICE
        num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(binary_mask, connectivity=8)
        labels = (labels > threshold).astype(np.uint8)

        if recurring_patches is None:
            recurring_patches = labels
        else:
            # Find patches that appear more than twice and update the recurring patches mask
            non_background_labels = labels > 0
            recurring_patches[non_background_labels] += labels[non_background_labels]

        # Identify patches that appear more than twice in the year
        patches_more_than_twice = recurring_patches >= 2  

        # Label patches in the last image with the recurring patches
        last_image = files[-1]  
        with rasterio.open(os.path.join(viirs_denoise_path, last_image)) as src:
            last_viirs_image = src.read(1)
            labeled_image = np.zeros_like(last_viirs_image)
            labeled_image[patches_more_than_twice] = last_viirs_image[patches_more_than_twice]

        recurring_binary_mask = (labeled_image > threshold).astype(np.uint8)    

    for file in files:
        image_path = os.path.join(viirs_denoise_path, file)
        with rasterio.open(image_path) as src:
            viirs_image = src.read(1)  # Read band 1 as a NumPy array
            profile = src.profile
        
        binary_mask_viirs = (viirs_image > threshold).astype(np.uint8)
        copy_viirs = viirs_image.copy() 
        copy_viirs[:] = 0.0

        for i in range(np.max(labels) + 1):
            if i > 0:
                indices = np.where(labels == i)
                n = np.sum(recurring_binary_mask[indices])
                n_r = np.sum(binary_mask_viirs[indices])
                PR = (n_r / n) * 100
                
                if PR >= 40: 
                    copy_viirs[indices] = viirs_image[indices]
                else:
                    pass

        output_dir = raw_path + r'\output_post-processed'
        name = file.rstrip(".tif") + 'final.tif'
        output = os.path.join(output_dir, name)

        with rasterio.open(output, 'w', **profile) as dst:
            dst.write(copy_viirs, 1)